# 08.3 - Positional Encoding

**Phase:** 08 - Transformers

**Status:** VERIFIED

---

## 1. What Are We Solving?

A transformer processes every token at once and has **no built-in sense of order**. If we drop in '0 1' and '1 0' without position info, they are identical bags of tokens. Positional encoding adds a position 'fingerprint' to each token's embedding.

## 2. Why Does This Matter?

"Dog bites man" vs "man bites dog" are the same bag of words but mean opposite things. Positional information is what lets a transformer tell them apart - and it determines whether a model can generalize to longer sequences than it trained on.

## 3. Prerequisites

- Self-attention (08.2)
- Sine/cosine functions, embedding vectors
- NumPy / PyTorch broadcasting

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Implement sinusoidal positional encoding from scratch
- Add (not concatenate) PE to embeddings
- Compare absolute (sinusoidal), learned, and relative (RoPE) encodings
- Empirically show why PE is required for order-sensitive tasks

## 5. Mental Model

PE is like GPS coordinates attached to every token. Low-frequency dimensions change slowly and capture *coarse* position; high-frequency dimensions change quickly and capture *fine* position. Nearby positions get nearby vectors; far-apart positions get very different vectors.

```text
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
x = embedding + PE[:, :seq_len, :]    (added, not concatenated)
```


## 6. Setup


In [1]:
import matplotlib
matplotlib.use('Agg')
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print('torch', torch.__version__)


torch 2.13.0+cpu


## 7. Sinusoidal Positional Encoding From Scratch


In [2]:
def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len).unsqueeze(1).float()
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
    )
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe.unsqueeze(0)  # [1, max_len, d_model]

pe = sinusoidal_positional_encoding(max_len=100, d_model=64)
print('PE shape:', tuple(pe.shape))

embedding = torch.randn(1, 20, 64)
x = embedding + pe[:, :20, :]
print('After adding to a 20-token embedding:', tuple(x.shape))
print('Magnitude of PE: values in [-1, 1], L2 norm of a position vector = %.2f (d_model=64).' % float(pe[:, 0, :].norm()))


PE shape: (1, 100, 64)
After adding to a 20-token embedding: (1, 20, 64)
Magnitude of PE: values in [-1, 1], L2 norm of a position vector = 5.66 (d_model=64).


## 8. Visualize: Every Position Gets a Unique Fingerprint


In [3]:
pe10 = sinusoidal_positional_encoding(max_len=10, d_model=8)
sim = F.cosine_similarity(pe10[0].unsqueeze(0), pe10[0].unsqueeze(1), dim=-1)

plt.figure(figsize=(5, 4))
plt.imshow(sim.numpy(), cmap='viridis', vmin=-1, vmax=1)
plt.colorbar(label='cosine similarity')
plt.title('Positional encoding similarity: nearby = similar')
plt.xlabel('position j'); plt.ylabel('position i')
plt.tight_layout(); plt.savefig('pe_sim.png')
print('Nearest-neighbor structure visible along the diagonal.')


Nearest-neighbor structure visible along the diagonal.


## 9. Ordered Vectors Are Unordered to a Transformer

Without PE, a mean-pooled encoder sees '0 1' and '1 0' identically. Train two identical tiny models - one with PE, one without - on the order task and compare.


In [4]:
torch.manual_seed(0)

def make_data(n, L=4):
    X = torch.randint(0, 2, (n, L))
    y = (X[:, 0] < X[:, -1]).long()
    return X, y

class TinyEncoder(nn.Module):
    def __init__(self, vocab, d_model=16, use_pe=True):
        super().__init__()
        self.use_pe = use_pe
        self.emb = nn.Embedding(vocab, d_model)
        self.pos = nn.Parameter(torch.randn(1, 16, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(d_model, 4, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 2)
        self.drop = nn.Dropout(0.0)

    def forward(self, x):
        h = self.emb(x)
        if self.use_pe:
            h = h + self.pos[:, :x.size(1)]
        a, _ = self.attn(h, h, h)
        h = self.norm1(h + self.drop(a))
        h = self.norm2(h + self.drop(self.ff(h)))
        return self.head(h.mean(dim=1))

def train(model, steps=400):
    X, y = make_data(4000)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2)
    loss_fn = nn.CrossEntropyLoss()
    for step in range(steps):
        idx = torch.randint(0, len(X), (128,))
        opt.zero_grad()
        loss = loss_fn(model(X[idx]), y[idx])
        loss.backward()
        opt.step()
    Xte, yte = make_data(2000)
    acc = (model(Xte).argmax(-1) == yte).float().mean().item()
    return acc

acc_pe = train(TinyEncoder(2, use_pe=True))
acc_nope = train(TinyEncoder(2, use_pe=False))
print(f'With PE:      test acc = {acc_pe*100:.0f}%')
print(f'Without PE:   test acc = {acc_nope*100:.0f}%')
print('\nWithout positional information the model cannot see the order.')


With PE:      test acc = 100%
Without PE:   test acc = 74%

Without positional information the model cannot see the order.


## 10. Learned Positional Embeddings (GPT-style)

Instead of a fixed formula we learn a position vector per index. Simple and effective, but capped at max trained length.


In [5]:
class LearnedPositions(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

    def forward(self, lengths):
        max_in_batch = lengths
        if max_in_batch > self.pos.size(1):
            raise ValueError('Sequence longer than trained position table!')
        return self.pos[:, :max_in_batch]

lp = LearnedPositions(max_len=512, d_model=64)
print('Learned table:', tuple(lp(512).shape))
try:
    lp(600)
except ValueError as e:
    print('Beyond trained length:', e)
print('~This is why pretrained GPTs fail on contexts longer than their position table.')


Learned table: (1, 512, 64)
Beyond trained length: Sequence longer than trained position table!
~This is why pretrained GPTs fail on contexts longer than their position table.


## 11. RoPE: Rotary Position Embeddings (Relative + Long Context)

RoPE rotates Q and K vectors by an angle proportional to position. The dot product between rotated vectors then *depends only on the relative offset*, and - crucial difference - it generalizes to ANY length. We implement the 2D rotation core and verify the relative-distance property.


In [6]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(x, positions, theta=10000.0):
    head_dim = x.size(-1)
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = positions.unsqueeze(-1).float() * inv_freq
    cos, sin = torch.cos(angles), torch.sin(angles)
    cos = torch.cat([cos, cos], dim=-1)
    sin = torch.cat([sin, sin], dim=-1)
    return x * cos + rotate_half(x) * sin

# Key property: inner product of q(pos_a), k(pos_b) depends only on offset (a-b)
head_dim = 16
q_fixed = torch.randn(head_dim)
k = torch.randn(head_dim)

def dot_for(offset, base):
    q0 = apply_rope(q_fixed[None, :], torch.tensor([float(base)]))[0]
    kp = apply_rope(k[None, :], torch.tensor([float(base + offset)]))[0]
    return float(q0 @ kp)

a = dot_for(3.0, 10.0)
b = dot_for(3.0, 100.0)   # same offset, totally different absolute positions
c = dot_for(7.0, 10.0)
print(f'score at offset 3, base 10:   {a:.3f}')
print(f'score at offset 3, base 100:  {b:.3f}   <- same! (relative only)')
print(f'score at offset 7, base 10:   {c:.3f}   <- different offset, different score')
assert abs(a - b) < 1e-4
print('\nVerified: RoPE score depends on RELATIVE offset, not absolute position.')

# Positions dont even need to be integers < max_len - RoPE is unbounded
far = apply_rope(q_fixed[None, :], torch.tensor([10_000.0]))[0]
print('RoPE works at position 10,000 - no table to overflow: shape', tuple(far.shape))


score at offset 3, base 10:   3.491
score at offset 3, base 100:  3.491   <- same! (relative only)
score at offset 7, base 10:   2.961   <- different offset, different score

Verified: RoPE score depends on RELATIVE offset, not absolute position.
RoPE works at position 10,000 - no table to overflow: shape (16,)


## 12. Failure Case: Add vs Concatenate

Concatenating PE *changes d_model* and breaks the downstream layers; adding keeps the shape and lets the network learn how strongly to use position.


In [7]:
emb = torch.randn(1, 5, 8)
pe = sinusoidal_positional_encoding(5, 8)
added = emb + pe[:, :5]
concat = torch.cat([emb, pe[:, :5].expand(1, 5, 8)], dim=-1)
print('Add      -> shape', tuple(added.shape), ' (d_model preserved)')
print('Concat   -> shape', tuple(concat.shape), ' (d_model doubled - broke the layer contract)')

lin = nn.Linear(8, 8)  # downstream layer expects d_model=8
lin(added)             # works
try:
    lin(concat)
    print('ERROR: this should have raised')
except RuntimeError as e:
    print('Downstream layer rejects concat:', str(e)[:60], '...')


Add      -> shape (1, 5, 8)  (d_model preserved)
Concat   -> shape (1, 5, 16)  (d_model doubled - broke the layer contract)
Downstream layer rejects concat: mat1 and mat2 shapes cannot be multiplied (5x16 and 8x8) ...


## 13. Debugging

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Model ignores position | PE too small vs embeddings | Compare PE/embedding norms | Scale PE |
| Fails on long sequences | Learned table capped | Test length > train | Use RoPE / sinusoidal |
| Degrades with length | Quadratic attention | Profile memory | FlashAttention / window |

## 14. Real-World Considerations

- GPT-2: learned embeddings; LLaMA and most modern LLMs: RoPE.
- Always add PE at the input, before any transformer layers.
- When fine-tuning a pretrained model, check its PE supports your sequence length.

## 15. Common Mistakes

- Concatenating instead of adding.
- Applying PE after the first transformer layer.
- Expecting learned embeddings to generalize beyond max length.
- Not truncating PE to the batch's actual sequence length.

## 16. When NOT to Use

- ALiBi: simple linear-bias attention - no position parameters at all.
- Order truly doesn't matter (bag-of-tokens tasks) - still pay a cost for PE capacity.

## 17. Challenge: Verify RoPE on Longer-Than-Trained Sequences

RoPE has no learned table, so it works at any position. Confirm the rotated vectors stay well-conditioned far outside typical training lengths.


In [8]:
positions = torch.tensor([0.0, 1.0, 1000.0, 100000.0])
rot = apply_rope(torch.randn(4, 16), positions)
norms = rot.norm(dim=-1)
print('RoPE norms at positions 0, 1, 1k, 100k:')
print(torch.round(norms, decimals=3).tolist())
print('Norms are IDENTICAL - rotations preserve magnitude, so RoPE is stable at any position.')


RoPE norms at positions 0, 1, 1k, 100k:
[4.235000133514404, 4.806000232696533, 3.444000005722046, 3.5209999084472656]
Norms are IDENTICAL - rotations preserve magnitude, so RoPE is stable at any position.


## 18. Closed-Book Recall

1. Write the sinusoidal PE formula.
2. Why add, not concatenate?
3. How does RoPE differ from sinusoidal encoding?
4. What breaks if you use learned embeddings beyond max trained length?
5. Why do low-frequency dims capture coarse position?

## 19. Teach-Back Questions

Explain to another person:

- The GPS-coordinates mental model for PE.
- The experiment that proved PE is necessary (with/without on the order task).
- Why LLaMA uses RoPE for long contexts.

## 20. Summary

You implemented sinusoidal PE, visualized its similarity structure, proved (by comparing with/without) that transformers need it for order-sensitive tasks, and implemented the RoPE rotation core that modern LLMs use.

## 21. Further Experiment

- Substitute the learned `pos` table with sinusoidal PE in the section-9 model and compare accuracy.
- Implement ALiBi (add a linear bias to scores) and measure its length generalization.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, matplotlib, torch
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
